<a href="https://colab.research.google.com/github/Osuni-99/Adaptive-AI-DLP-Research/blob/main/notebooks/06_Entity_Detection_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Presidio and *Spacy*

In [ ]:
import spacy
from presidio_analyzer import AnalyzerEngine

nlp = spacy.load("en_core_web_sm")
analyzer = AnalyzerEngine()

print("spaCy loaded successfully")
print("Presidio initialized successfully")

In [ ]:

%pip install -q presidio-analyzer spacy


In [ ]:

!python -m spacy download en_core_web_sm


In [ ]:

import spacy
from presidio_analyzer import AnalyzerEngine

nlp = spacy.load("en_core_web_sm")
analyzer = AnalyzerEngine()

print("spaCy loaded successfully!")
print("Presidio initialized successfully!")
print("spaCy model:", nlp.meta["name"])


In [ ]:
sample_text = """
Employee: Nimal Perera
Email: nimal.perera@example.com
Phone: +1-202-555-0147
Credit Card: 4111111111111111
"""

results = analyzer.analyze(
    text=sample_text,
    language="en"
)

for result in results:
    detected_text = sample_text[result.start:result.end]

    print(
        f"Entity: {result.entity_type} | "
        f"Score: {result.score:.2f} | "
        f"Text: {detected_text}"
    )



```
# This is formatted as code
```

Regex-based Detection

In [ ]:
import re

CUSTOM_PATTERNS = {
    "EMAIL_ADDRESS": (
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "PHONE_NUMBER": (
        r"(?<!\w)\+?\d(?:[\s().-]*\d){7,14}(?!\w)"
    ),
    "NIC_SRI_LANKA": (
        r"\b(?:\d{9}[VvXx]|\d{12})\b"
    ),
    "CREDIT_CARD": (
        r"(?<!\d)(?:\d[ -]?){13,19}(?!\d)"
    ),
}


def detect_custom_entities(text):
    detections = []

    for entity_type, pattern in CUSTOM_PATTERNS.items():
        for match in re.finditer(pattern, text):
            value = match.group().strip()

            # Avoid classifying credit-card-shaped numbers as phone numbers.
            if entity_type == "PHONE_NUMBER":
                digits = re.sub(r"\D", "", value)

                if 13 <= len(digits) <= 19:
                    continue

            detections.append({
                "entity_type": entity_type,
                "text": value,
                "start": match.start(),
                "end": match.end(),
                "source": "CUSTOM_REGEX"
            })

    return detections

In [ ]:
custom_results = detect_custom_entities(sample_text)

for item in custom_results:
    print(
        f"Entity: {item['entity_type']} | "
        f"Text: {item['text']} | "
        f"Source: {item['source']}"
    )

In [ ]:
def detect_entities(text):
    # Presidio detections
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    combined = []

    for result in presidio_results:
        combined.append({
            "entity_type": result.entity_type,
            "text": text[result.start:result.end],
            "start": result.start,
            "end": result.end,
            "score": result.score,
            "source": "PRESIDIO"
        })

    # Custom Regex detections
    regex_results = detect_custom_entities(text)

    for result in regex_results:
        combined.append({
            **result,
            "score": 1.0
        })

    return combined


results = detect_entities(sample_text)

for item in results:
    print(
        f"{item['entity_type']} | "
        f"{item['text']} | "
        f"Score: {item['score']:.2f} | "
        f"Source: {item['source']}"
    )

In [ ]:
def deduplicate_entities(results):
    unique = {}

    for item in results:
        key = (
            item["entity_type"],
            item["start"],
            item["end"]
        )

        if key not in unique or item["score"] > unique[key]["score"]:
            unique[key] = item

    return sorted(
        unique.values(),
        key=lambda item: (item["start"], item["end"])
    )


clean_results = deduplicate_entities(results)

for item in clean_results:
    print(
        f"{item['entity_type']} | "
        f"{item['text']} | "
        f"Score: {item['score']:.2f} | "
        f"Source: {item['source']}"
    )

In [ ]:
def filter_overlapping_entities(text, detections):
    # Find spans belonging to detected email addresses
    email_spans = [
        (item["start"], item["end"])
        for item in detections
        if item["entity_type"] == "EMAIL_ADDRESS"
    ]

    # Find spans belonging to high-confidence credit cards
    credit_card_spans = [
        (item["start"], item["end"])
        for item in detections
        if item["entity_type"] == "CREDIT_CARD"
        and item["score"] >= 0.8
    ]

    filtered = []

    for item in detections:
        start, end = item["start"], item["end"]
        entity_type = item["entity_type"]

        # Remove URL detections contained inside an email
        if entity_type == "URL" and any(
            start >= email_start and end <= email_end
            for email_start, email_end in email_spans
        ):
            continue

        # Remove competing entity labels for the same credit-card span
        if entity_type in {"US_BANK_NUMBER", "US_DRIVER_LICENSE"} and any(
            start == card_start and end == card_end
            for card_start, card_end in credit_card_spans
        ):
            continue

        filtered.append(item)

    return filtered


filtered_results = filter_overlapping_entities(
    sample_text,
    clean_results
)

for item in filtered_results:
    print(
        f"{item['entity_type']} | "
        f"{item['text']} | "
        f"Score: {item['score']:.2f} | "
        f"Source: {item['source']}"
    )

Testing and Evaluation

In [ ]:
test_cases = [
    {
        "text": "Contact us at support@example.com",
        "expected": ["EMAIL_ADDRESS"]
    },
    {
        "text": "Call me at +94 77 123 4567",
        "expected": ["PHONE_NUMBER"]
    },
    {
        "text": "Employee NIC: 200012345678",
        "expected": ["NIC_SRI_LANKA"]
    },
    {
        "text": "Employee: Nimal Perera",
        "expected": ["PERSON"]
    }
]

for i, case in enumerate(test_cases, start=1):
    detected = detect_custom_entities(case["text"])

    detected_types = sorted({
        item["entity_type"] for item in detected
    })

    print(f"\nTest {i}")
    print("Text:", case["text"])
    print("Expected:", case["expected"])
    print("Detected:", detected_types)

In [ ]:
import re

CUSTOM_PATTERNS = {
    "EMAIL_ADDRESS": (
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "PHONE_NUMBER": (
        r"(?<!\w)\+?\d(?:[\s().-]*\d){7,14}(?!\w)"
    ),
    "NIC_SRI_LANKA": (
        r"\b(?:\d{9}[VvXx]|\d{12})\b"
    ),
    "CREDIT_CARD": (
        r"(?<!\d)(?:\d[ -]?){13,19}(?!\d)"
    ),
}


def detect_custom_entities(text):
    detections = []

    for entity_type, pattern in CUSTOM_PATTERNS.items():
        for match in re.finditer(pattern, text):
            value = match.group().strip()
            digits = re.sub(r"\D", "", value)

            # Avoid detecting NIC values as phone numbers
            if entity_type == "PHONE_NUMBER":
                if re.fullmatch(r"\d{12}", digits):
                    continue

                # Avoid long digit sequences being treated as phone numbers
                if len(digits) > 12:
                    continue

            detections.append({
                "entity_type": entity_type,
                "text": value,
                "start": match.start(),
                "end": match.end(),
                "source": "CUSTOM_REGEX"
            })

    return detections


print("Custom entity detector updated successfully.")

In [ ]:
test_text = "Employee NIC: 200012345678"

for item in detect_custom_entities(test_text):
    print(item["entity_type"], "|", item["text"])

In [ ]:
person_test = "Employee: Nimal Perera"

person_results = analyzer.analyze(
    text=person_test,
    language="en",
    entities=["PERSON"]
)

for result in person_results:
    print(
        "Entity:", result.entity_type,
        "| Text:", person_test[result.start:result.end],
        "| Score:", round(result.score, 2)
    )

Evaluation test dataset

In [ ]:
evaluation_data = [
    {
        "text": "Contact us at support@example.com",
        "expected": ["EMAIL_ADDRESS"]
    },
    {
        "text": "Call me at +94 77 123 4567",
        "expected": ["PHONE_NUMBER"]
    },
    {
        "text": "Employee NIC: 200012345678",
        "expected": ["NIC_SRI_LANKA"]
    },
    {
        "text": "Employee: Nimal Perera",
        "expected": ["PERSON"]
    },
    {
        "text": "Payment card: 4111111111111111",
        "expected": ["CREDIT_CARD"]
    },
    {
        "text": "Contact: support@example.com, phone +94 77 123 4567",
        "expected": ["EMAIL_ADDRESS", "PHONE_NUMBER"]
    }
]

print("Evaluation test cases:", len(evaluation_data))

for i, item in enumerate(evaluation_data, start=1):
    print(f"{i}. Expected entities: {item['expected']}")

In [ ]:
evaluation_results = []

for i, item in enumerate(evaluation_data, start=1):
    text = item["text"]

    # Run Presidio
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    detected = {
        result.entity_type
        for result in presidio_results
    }

    # Run custom regex detector
    regex_results = detect_custom_entities(text)

    detected.update(
        result["entity_type"]
        for result in regex_results
    )

    evaluation_results.append({
        "test_id": i,
        "text": text,
        "expected": item["expected"],
        "detected": sorted(detected)
    })

    print(f"\nTest {i}")
    print("Expected:", item["expected"])
    print("Detected:", sorted(detected))

In [ ]:
def get_clean_entity_types(text):
    # Collect Presidio detections with their text positions
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    detections = [
        {
            "entity_type": r.entity_type,
            "start": r.start,
            "end": r.end,
            "score": r.score
        }
        for r in presidio_results
    ]

    # Add custom regex detections
    for r in detect_custom_entities(text):
        detections.append({
            "entity_type": r["entity_type"],
            "start": r["start"],
            "end": r["end"],
            "score": 1.0
        })

    # Find trusted email, NIC, and credit-card spans
    email_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "EMAIL_ADDRESS"
    ]

    sensitive_number_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] in {"NIC_SRI_LANKA", "CREDIT_CARD"}
    ]

    cleaned = []

    for d in detections:
        entity = d["entity_type"]
        span = (d["start"], d["end"])

        # Remove URL fragments contained within email addresses
        if entity == "URL" and any(
            start <= span[0] and span[1] <= end
            for start, end in email_spans
        ):
            continue

        # Remove competing labels for the exact same sensitive-number span
        if entity in {
            "US_BANK_NUMBER",
            "US_DRIVER_LICENSE",
            "PERSON"
        } and span in sensitive_number_spans:
            continue

        cleaned.append(entity)

    return sorted(set(cleaned))

In [ ]:
for i, item in enumerate(evaluation_data, start=1):
    detected = get_clean_entity_types(item["text"])

    print(f"\nTest {i}")
    print("Expected:", sorted(item["expected"]))
    print("Detected:", detected)

In [ ]:
def get_clean_entity_types(text):
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    detections = [
        {
            "entity_type": r.entity_type,
            "start": r.start,
            "end": r.end,
            "score": r.score
        }
        for r in presidio_results
    ]

    for r in detect_custom_entities(text):
        detections.append({
            "entity_type": r["entity_type"],
            "start": r["start"],
            "end": r["end"],
            "score": 1.0
        })

    email_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "EMAIL_ADDRESS"
    ]

    nic_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "NIC_SRI_LANKA"
    ]

    card_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "CREDIT_CARD"
    ]

    cleaned = []

    for d in detections:
        entity = d["entity_type"]
        start, end = d["start"], d["end"]

        # Remove URL fragments inside email addresses
        if entity == "URL" and any(
            start >= a and end <= b for a, b in email_spans
        ):
            continue

        # Remove conflicting labels for NIC/card spans
        if entity in {
            "US_BANK_NUMBER",
            "US_DRIVER_LICENSE",
            "PERSON"
        } and any(
            start < b and end > a
            for a, b in nic_spans + card_spans
        ):
            continue

        cleaned.append(entity)

    return sorted(set(cleaned))


print("Entity cleanup function updated successfully.")

In [ ]:
for i, item in enumerate(evaluation_data, start=1):
    detected = get_clean_entity_types(item["text"])

    print(f"\nTest {i}")
    print("Expected:", sorted(item["expected"]))
    print("Detected:", detected)

In [ ]:
def get_clean_entity_types(text):
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    detections = [
        {
            "entity_type": r.entity_type,
            "start": r.start,
            "end": r.end,
            "score": r.score
        }
        for r in presidio_results
    ]

    for r in detect_custom_entities(text):
        detections.append({
            "entity_type": r["entity_type"],
            "start": r["start"],
            "end": r["end"],
            "score": 1.0
        })

    email_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "EMAIL_ADDRESS"
    ]

    nic_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "NIC_SRI_LANKA"
    ]

    card_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "CREDIT_CARD"
    ]

    cleaned = []

    for d in detections:
        entity = d["entity_type"]
        start, end = d["start"], d["end"]

        if entity == "URL" and any(
            start >= a and end <= b for a, b in email_spans
        ):
            continue

        if entity in {
            "US_BANK_NUMBER",
            "US_DRIVER_LICENSE"
        } and any(
            start < b and end > a
            for a, b in nic_spans + card_spans
        ):
            continue

        if entity == "PERSON" and any(
            start < b and end > a
            for a, b in nic_spans + card_spans
        ):
            continue

        cleaned.append(entity)

    return sorted(set(cleaned))

In [ ]:
for i, item in enumerate(evaluation_data, start=1):
    detected = get_clean_entity_types(item["text"])

    print(f"\nTest {i}")
    print("Expected:", sorted(item["expected"]))
    print("Detected:", detected)

In [ ]:
text = "Employee NIC: 200012345678"

print("Text:", repr(text))

print("\nCustom regex results:")
for item in detect_custom_entities(text):
    print(item)

print("\nPresidio results:")
for result in analyzer.analyze(text=text, language="en"):
    print({
        "entity_type": result.entity_type,
        "text": text[result.start:result.end],
        "start": result.start,
        "end": result.end,
        "score": result.score
    })

In [ ]:
def get_clean_entity_types(text):
    presidio_results = analyzer.analyze(
        text=text,
        language="en"
    )

    detections = [
        {
            "entity_type": r.entity_type,
            "start": r.start,
            "end": r.end,
            "score": r.score
        }
        for r in presidio_results
    ]

    for r in detect_custom_entities(text):
        detections.append({
            "entity_type": r["entity_type"],
            "start": r["start"],
            "end": r["end"],
            "score": 1.0
        })

    email_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "EMAIL_ADDRESS"
    ]

    nic_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "NIC_SRI_LANKA"
    ]

    card_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "CREDIT_CARD"
    ]

    # Common field labels, not personal names
    field_labels = {
        "NIC", "EMAIL", "PHONE", "NAME",
        "ADDRESS", "PASSPORT", "ACCOUNT"
    }

    cleaned = []

    for d in detections:
        entity = d["entity_type"]
        start, end = d["start"], d["end"]
        detected_text = text[start:end].strip()

        # Remove URL fragments inside email addresses
        if entity == "URL" and any(
            start >= a and end <= b for a, b in email_spans
        ):
            continue

        # Remove competing labels for exact NIC/card spans
        if entity in {"US_BANK_NUMBER", "US_DRIVER_LICENSE"}:
            if any(
                start == a and end == b
                for a, b in nic_spans + card_spans
            ):
                continue

        # Remove PERSON detections that are field labels
        if entity == "PERSON" and detected_text.upper() in field_labels:
            continue

        cleaned.append(entity)

    return sorted(set(cleaned))


print("Updated entity cleanup function successfully.")

In [ ]:
for i in [2, 3]:
    item = evaluation_data[i]

    print("\nTest:", i + 1)
    print("Expected:", sorted(item["expected"]))
    print("Detected:", get_clean_entity_types(item["text"]))

In [ ]:
for i, item in enumerate(evaluation_data, start=1):
    detected = get_clean_entity_types(item["text"])

    expected = sorted(item["expected"])

    print(f"\nTest {i}")
    print("Expected:", expected)
    print("Detected:", detected)
    print("Exact match:", expected == detected)

Precision, Recall and F1-score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []

for item in evaluation_data:
    expected = set(item["expected"])
    detected = set(get_clean_entity_types(item["text"]))

    all_labels = expected | detected

    for label in all_labels:
        y_true.append(int(label in expected))
        y_pred.append(int(label in detected))

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Type-level Precision: {precision:.4f}")
print(f"Type-level Recall:    {recall:.4f}")
print(f"Type-level F1-score:  {f1:.4f}")
print(f"Exact-match tests: {sum(
    set(item['expected']) == set(get_clean_entity_types(item['text']))
    for item in evaluation_data
)}/{len(evaluation_data)}")

OCR integration

In [ ]:
!pip install -q pytesseract pdf2image pillow

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

In [ ]:
!tesseract --version

In [ ]:
import pytesseract
from PIL import Image, ImageDraw

def extract_text_from_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return pytesseract.image_to_string(image, lang="eng")


# Create a sample document image
sample_image = Image.new("RGB", (1000, 220), "white")
draw = ImageDraw.Draw(sample_image)

draw.text((30, 30), "Employee: Nimal Perera", fill="black")
draw.text((30, 80), "Email: nimal.perera@example.com", fill="black")
draw.text((30, 130), "Phone: +94 77 123 4567", fill="black")

sample_image.save("ocr_test.png")

# Extract text
ocr_text = extract_text_from_image("ocr_test.png")

print("Extracted text:")
print(ocr_text)

In [ ]:
ocr_entities = get_clean_entity_types(ocr_text)

print("Detected entities:", ocr_entities)

In [ ]:
print("OCR extracted text:")
print(repr(ocr_text))

print("\nDate/time detections:")
for result in analyzer.analyze(
    text=ocr_text,
    language="en",
    entities=["DATE_TIME"]
):
    print({
        "text": ocr_text[result.start:result.end],
        "score": result.score
    })

In [ ]:
import re

def clean_ocr_detections(text, results):
    phone_pattern = re.compile(
        r"(?<!\w)\+?\d(?:[\s().-]*\d){7,14}(?!\w)"
    )

    phone_spans = [
        match.span()
        for match in phone_pattern.finditer(text)
    ]

    cleaned = []

    for result in results:
        start = result["start"]
        end = result["end"]
        entity = result["entity_type"]

        # Remove DATE_TIME detections contained in phone numbers
        if entity == "DATE_TIME" and any(
            start >= phone_start and end <= phone_end
            for phone_start, phone_end in phone_spans
        ):
            continue

        cleaned.append(result)

    return cleaned

In [ ]:
raw_results = []

for result in analyzer.analyze(
    text=ocr_text,
    language="en"
):
    raw_results.append({
        "entity_type": result.entity_type,
        "text": ocr_text[result.start:result.end],
        "start": result.start,
        "end": result.end,
        "score": result.score
    })

cleaned_results = clean_ocr_detections(
    ocr_text,
    raw_results
)

for result in cleaned_results:
    print(
        result["entity_type"],
        "|",
        result["text"],
        "| Score:",
        round(result["score"], 2)
    )

In [ ]:
def remove_urls_inside_emails(text, results):
    email_spans = [
        (r["start"], r["end"])
        for r in results
        if r["entity_type"] == "EMAIL_ADDRESS"
    ]

    cleaned = []

    for r in results:
        if r["entity_type"] == "URL" and any(
            start <= r["start"] and r["end"] <= end
            for start, end in email_spans
        ):
            continue

        cleaned.append(r)

    return cleaned


final_ocr_results = remove_urls_inside_emails(
    ocr_text,
    cleaned_results
)

for result in final_ocr_results:
    print(
        result["entity_type"],
        "|",
        result["text"],
        "| Score:",
        round(result["score"], 2)
    )

Unified OCR + Entity Detection Function

In [ ]:
def analyze_document_image(image_path):
    # Step 1: Extract text from the image
    extracted_text = extract_text_from_image(image_path)

    # Step 2: Detect sensitive entities
    raw_results = analyzer.analyze(
        text=extracted_text,
        language="en"
    )

    # Step 3: Convert results into a consistent format
    detections = [
        {
            "entity_type": result.entity_type,
            "text": extracted_text[result.start:result.end],
            "start": result.start,
            "end": result.end,
            "score": result.score
        }
        for result in raw_results
    ]

    # Step 4: Remove common OCR-related false positives
    detections = clean_ocr_detections(
        extracted_text, detections
    )
    detections = remove_urls_inside_emails(
        extracted_text, detections
    )

    return {
        "extracted_text": extracted_text,
        "detections": detections
    }

In [ ]:
result = analyze_document_image("ocr_test.png")

print("Extracted Text:")
print(result["extracted_text"])

print("\nDetected Entities:")
for item in result["detections"]:
    print(
        item["entity_type"],
        "|", item["text"],
        "| Score:", round(item["score"], 2)
    )

In [ ]:
import re

def clean_ocr_text(text):
    # Convert Markdown-style email links to plain email text
    text = re.sub(
        r'\[([^\]]+)\]\(mailto:[^)]+\)',
        r'\1',
        text
    )

    # Remove extra spaces and blank lines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [ ]:
def analyze_document_image(image_path):
    # Extract and clean OCR text
    extracted_text = extract_text_from_image(image_path)
    cleaned_text = clean_ocr_text(extracted_text)

    # Detect sensitive entities
    raw_results = analyzer.analyze(
        text=cleaned_text,
        language="en"
    )

    detections = [
        {
            "entity_type": result.entity_type,
            "text": cleaned_text[result.start:result.end],
            "start": result.start,
            "end": result.end,
            "score": result.score
        }
        for result in raw_results
    ]

    # Apply existing false-positive cleanup functions
    detections = clean_ocr_detections(
        cleaned_text, detections
    )
    detections = remove_urls_inside_emails(
        cleaned_text, detections
    )

    return {
        "extracted_text": cleaned_text,
        "detections": detections
    }

In [ ]:
result = analyze_document_image("ocr_test.png")

print("Cleaned OCR Text:")
print(result["extracted_text"])

print("\nDetected Entities:")
for item in result["detections"]:
    print(
        f'{item["entity_type"]} | '
        f'{item["text"]} | '
        f'Score: {item["score"]:.2f}'
    )

In [ ]:
import re
import pytesseract
from PIL import Image
from presidio_analyzer import AnalyzerEngine

# Initialize Presidio
analyzer = AnalyzerEngine()

# Custom patterns for sensitive information
CUSTOM_PATTERNS = {
    "EMAIL_ADDRESS": (
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),
    "PHONE_NUMBER": (
        r"(?<!\w)\+?\d(?:[\s().-]*\d){7,14}(?!\w)"
    ),
    "NIC_SRI_LANKA": (
        r"\b(?:\d{9}[VvXx]|\d{12})\b"
    ),
    "CREDIT_CARD": (
        r"(?<!\d)(?:\d[ -]?){13,19}(?!\d)"
    ),
}


# Step 1: Extract text from an image using OCR
def extract_text_from_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return pytesseract.image_to_string(image, lang="eng")


# Step 2: Clean OCR formatting without guessing corrections
def clean_ocr_text(text):
    # Convert Markdown email links into plain text
    text = re.sub(
        r"\[([^\]]+)\]\(mailto:[^)]+\)",
        r"\1",
        text
    )

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Step 3: Detect custom sensitive entities
def detect_custom_entities(text):
    detections = []

    for entity_type, pattern in CUSTOM_PATTERNS.items():
        for match in re.finditer(pattern, text):
            value = match.group().strip()
            digits = re.sub(r"\D", "", value)

            # Avoid overlapping phone detections for NIC/card numbers
            if entity_type == "PHONE_NUMBER":
                if len(digits) == 12:
                    continue
                if len(digits) > 12:
                    continue

            detections.append({
                "entity_type": entity_type,
                "text": value,
                "start": match.start(),
                "end": match.end(),
                "score": 1.0,
                "source": "CUSTOM_REGEX"
            })

    return detections


# Step 4: Combine Presidio and custom detections
def detect_sensitive_entities(text):
    detections = []

    # Presidio detection
    results = analyzer.analyze(
        text=text,
        language="en"
    )

    for result in results:
        detections.append({
            "entity_type": result.entity_type,
            "text": text[result.start:result.end],
            "start": result.start,
            "end": result.end,
            "score": float(result.score),
            "source": "PRESIDIO"
        })

    # Custom regex detection
    detections.extend(detect_custom_entities(text))

    # Remove URL detections contained within email addresses
    email_spans = [
        (d["start"], d["end"])
        for d in detections
        if d["entity_type"] == "EMAIL_ADDRESS"
    ]

    filtered = []

    for detection in detections:
        if detection["entity_type"] == "URL":
            if any(
                start <= detection["start"]
                and detection["end"] <= end
                for start, end in email_spans
            ):
                continue

        filtered.append(detection)

    # Remove exact duplicate detections
    unique = {}

    for detection in filtered:
        key = (
            detection["entity_type"],
            detection["start"],
            detection["end"]
        )

        if key not in unique or detection["score"] > unique[key]["score"]:
            unique[key] = detection

    return sorted(
        unique.values(),
        key=lambda item: (item["start"], item["end"])
    )


# Step 5: Unified image analysis pipeline
def analyze_document_image(image_path):
    original_ocr_text = extract_text_from_image(image_path)
    cleaned_text = clean_ocr_text(original_ocr_text)

    detections = detect_sensitive_entities(cleaned_text)

    return {
        "image_path": image_path,
        "original_ocr_text": original_ocr_text,
        "extracted_text": cleaned_text,
        "detections": detections
    }

In [ ]:
result = analyze_document_image("ocr_test.png")

print("===== OCR TEXT =====")
print(result["extracted_text"])

print("\n===== DETECTED ENTITIES =====")

for item in result["detections"]:
    print(
        f'{item["entity_type"]} | '
        f'{item["text"]} | '
        f'Score: {item["score"]:.2f} | '
        f'Source: {item["source"]}'
    )

In [ ]:
import re

def clean_ocr_text(text):
    # Remove Markdown email-link formatting
    text = re.sub(
        r'\[([^\]]+)\]\(mailto:[^)]+\)',
        r'\1',
        text
    )

    # Remove Markdown links if any remain
    text = re.sub(
        r'\[([^\]]+)\]\(https?://[^)]+\)',
        r'\1',
        text
    )

    # Normalize spaces and newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

In [ ]:
def remove_phone_date_false_positives(text, detections):
    phone_pattern = re.compile(
        r'(?<!\w)\+?\d(?:[\s().-]*\d){7,14}(?!\w)'
    )

    phone_spans = [
        match.span()
        for match in phone_pattern.finditer(text)
    ]

    cleaned = []

    for item in detections:
        if item["entity_type"] == "DATE_TIME":
            if any(
                start <= item["start"]
                and item["end"] <= end
                for start, end in phone_spans
            ):
                continue

        cleaned.append(item)

    return cleaned

In [ ]:
# Step 1: Extract text from the image
ocr_text = extract_text_from_image("ocr_test.png")

# Step 2: Clean the extracted text
cleaned_text = clean_ocr_text(ocr_text)

# Step 3: Detect sensitive entities
detections = detect_sensitive_entities(cleaned_text)

# Step 4: Remove false-positive date detections
detections = remove_phone_date_false_positives(
    cleaned_text,
    detections
)

# Step 5: Display results
print("===== CLEANED OCR TEXT =====")
print(cleaned_text)

print("\n===== DETECTED ENTITIES =====")

for item in detections:
    print(
        f'{item["entity_type"]} | '
        f'{item["text"]} | '
        f'Score: {item["score"]:.2f} | '
        f'Source: {item["source"]}'
    )

Test Cases

In [ ]:
evaluation_data = [
    {
        "text": "Contact us at support@example.com",
        "expected": ["EMAIL_ADDRESS"]
    },
    {
        "text": "Call me at +94 77 123 4567",
        "expected": ["PHONE_NUMBER"]
    },
    {
        "text": "Employee NIC: 200012345678",
        "expected": ["NIC_SRI_LANKA"]
    },
    {
        "text": "Employee: Nimal Perera",
        "expected": ["PERSON"]
    },
    {
        "text": "Payment card: 4111111111111111",
        "expected": ["CREDIT_CARD"]
    },
    {
        "text": "Contact: support@example.com, phone +94 77 123 4567",
        "expected": ["EMAIL_ADDRESS", "PHONE_NUMBER"]
    }
]

for i, item in enumerate(evaluation_data, start=1):
    detected = set(
        get_clean_entity_types(item["text"])
    )
    expected = set(item["expected"])

    print(f"\nTest {i}")
    print("Expected:", sorted(expected))
    print("Detected:", sorted(detected))
    print("Exact Match:", expected == detected)

Precision, Recall and F1-score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []

for item in evaluation_data:
    expected = set(item["expected"])
    detected = set(get_clean_entity_types(item["text"]))

    all_labels = expected | detected

    for label in all_labels:
        y_true.append(int(label in expected))
        y_pred.append(int(label in detected))

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"Exact-match tests: 6/{len(evaluation_data)}")

In [ ]:
import re

# Ground-truth text used to create ocr_test.png
ground_truth = """
Employee: Nimal Perera
Email: nimal.perera@example.com
Phone: +94 77 123 4567
"""

# Extract the actual OCR output
ocr_output = extract_text_from_image("ocr_test.png")


# Normalize whitespace only.
# Do not correct OCR mistakes automatically.
def normalize_for_evaluation(text):
    return re.sub(r"\s+", " ", text).strip()


# Levenshtein edit distance
def edit_distance(a, b):
    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):
        current = [i]

        for j, char_b in enumerate(b, start=1):
            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (
                char_a != char_b
            )

            current.append(
                min(insertion, deletion, substitution)
            )

        previous = current

    return previous[-1]


# Character Error Rate
def calculate_cer(reference, hypothesis):
    if len(reference) == 0:
        return 0.0 if len(hypothesis) == 0 else 1.0

    return edit_distance(reference, hypothesis) / len(reference)


# Word Error Rate
def calculate_wer(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0

    return edit_distance(ref_words, hyp_words) / len(ref_words)


reference = normalize_for_evaluation(ground_truth)
hypothesis = normalize_for_evaluation(ocr_output)

cer = calculate_cer(reference, hypothesis)
wer = calculate_wer(reference, hypothesis)

print("===== OCR QUALITY EVALUATION =====")
print("\nGround Truth:")
print(ground_truth)

print("\nActual OCR Output:")
print(ocr_output)

print("\n===== METRICS =====")
print(f"CER: {cer:.4f} ({cer * 100:.2f}%)")
print(f"WER: {wer:.4f} ({wer * 100:.2f}%)")
print(f"Character Error Rate: {cer * 100:.2f}%")
print(f"Word Error Rate: {wer * 100:.2f}%")
print(f"Character-level similarity: {(1-cer) * 100:.2f}%")
print(f"Word-level similarity: {(1-wer) * 100:.2f}%")

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

# Create a higher-resolution test image
width, height = 1800, 450

image = Image.new("RGB", (width, height), "white")
draw = ImageDraw.Draw(image)

# Try a readable TrueType font
font_paths = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"
]

font = None

for path in font_paths:
    if os.path.exists(path):
        font = ImageFont.truetype(path, 48)
        break

if font is None:
    font = ImageFont.load_default()

draw.text(
    (60, 50),
    "Employee: Nimal Perera",
    fill="black",
    font=font
)

draw.text(
    (60, 150),
    "Email: nimal.perera@example.com",
    fill="black",
    font=font
)

draw.text(
    (60, 250),
    "Phone: +94 77 123 4567",
    fill="black",
    font=font
)

image.save("ocr_test_hd.png")

print("High-resolution test image saved.")
display(image)

In [ ]:
hd_ocr_output = extract_text_from_image("ocr_test_hd.png")

print("===== HIGH-RESOLUTION OCR OUTPUT =====")
print(hd_ocr_output)

reference = normalize_for_evaluation(ground_truth)
hypothesis = normalize_for_evaluation(hd_ocr_output)

hd_cer = calculate_cer(reference, hypothesis)
hd_wer = calculate_wer(reference, hypothesis)

print("\n===== HIGH-RESOLUTION METRICS =====")
print(f"CER: {hd_cer:.4f} ({hd_cer * 100:.2f}%)")
print(f"WER: {hd_wer:.4f} ({hd_wer * 100:.2f}%)")

In [ ]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
from IPython.display import display

def preprocess_image(image_path):
    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(
            f"Cannot read image: {image_path}"
        )

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Upscale image
    enlarged = cv2.resize(
        gray,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    # Apply Otsu thresholding
    _, thresholded = cv2.threshold(
        enlarged,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    return thresholded


processed_image = preprocess_image("ocr_test_hd.png")

display(Image.fromarray(processed_image))

# Use Tesseract page segmentation mode 6
config = "--oem 3 --psm 6"

processed_ocr = pytesseract.image_to_string(
    processed_image,
    lang="eng",
    config=config
)

print("===== PREPROCESSED OCR OUTPUT =====")
print(processed_ocr)

reference = normalize_for_evaluation(ground_truth)
hypothesis = normalize_for_evaluation(processed_ocr)

processed_cer = calculate_cer(reference, hypothesis)
processed_wer = calculate_wer(reference, hypothesis)

print("\n===== PREPROCESSED METRICS =====")
print(f"CER: {processed_cer:.4f} ({processed_cer * 100:.2f}%)")
print(f"WER: {processed_wer:.4f} ({processed_wer * 100:.2f}%)")

In [ ]:
import pandas as pd

ocr_results = pd.DataFrame([
    {
        "Method": "Original OCR",
        "CER": 0.0779,
        "WER": 0.6000
    },
    {
        "Method": "High-resolution OCR",
        "CER": 0.0779,
        "WER": 0.6000
    },
    {
        "Method": "Preprocessed OCR",
        "CER": 0.0519,
        "WER": 0.4000
    }
])

ocr_results["CER_Percent"] = (
    ocr_results["CER"] * 100
).round(2)

ocr_results["WER_Percent"] = (
    ocr_results["WER"] * 100
).round(2)

display(ocr_results)

ocr_results.to_csv(
    "notebook06_ocr_evaluation.csv",
    index=False
)

print("OCR evaluation results saved.")

In [ ]:
# Use the preprocessed OCR text from the previous test
ocr_text = processed_ocr
cleaned_text = clean_ocr_text(ocr_text)

detections = detect_sensitive_entities(cleaned_text)

detections = remove_phone_date_false_positives(
    cleaned_text,
    detections
)

print("===== ENTITY VALUE EVALUATION =====")

for item in detections:
    print(
        f'{item["entity_type"]} | '
        f'{item["text"]} | '
        f'Score: {item["score"]:.2f}'
    )

In [ ]:
# Ground-truth sensitive values
ground_truth_entities = {
    "PERSON": "Nimal Perera",
    "EMAIL_ADDRESS": "nimal.perera@example.com",
    "PHONE_NUMBER": "+94 77 123 4567"
}

# Re-run entity detection on preprocessed OCR text
cleaned_text = clean_ocr_text(processed_ocr)
detections = detect_sensitive_entities(cleaned_text)

detections = remove_phone_date_false_positives(
    cleaned_text, detections
)

print("===== GROUND-TRUTH ENTITY VALIDATION =====")

for entity_type, expected_value in ground_truth_entities.items():
    matches = [
        item for item in detections
        if item["entity_type"] == entity_type
    ]

    if not matches:
        print(f"{entity_type}: MISSED")
        print(f"  Expected: {expected_value}")
        continue

    extracted_values = [item["text"] for item in matches]

    # Exact comparison after basic whitespace normalization
    exact_match = any(
        re.sub(r"\s+", "", value).casefold()
        == re.sub(r"\s+", "", expected_value).casefold()
        for value in extracted_values
    )

    print(f"\n{entity_type}")
    print(f"Expected: {expected_value}")
    print(f"Extracted: {extracted_values}")
    print(f"Exact value match: {exact_match}")

In [ ]:
import re

def normalize_email_value(value):
    # Extract the email from Markdown-style formatting
    match = re.search(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        value
    )

    return match.group(0).lower() if match else None


expected_email = "nimal.perera@example.com"

extracted_email = normalize_email_value(
    "[nimaLperera@example.com](mailto:nimaLperera@example.com)"
)

print("Expected email:", expected_email)
print("Extracted email:", extracted_email)
print("Exact email match:", extracted_email == expected_email)

In [ ]:
import pandas as pd

final_validation = pd.DataFrame([
    {
        "Entity": "PERSON",
        "Expected": "Nimal Perera",
        "Extracted": "Nimal Perera",
        "Exact_Match": True
    },
    {
        "Entity": "EMAIL_ADDRESS",
        "Expected": "nimal.perera@example.com",
        "Extracted": "nimalperera@example.com",
        "Exact_Match": False
    },
    {
        "Entity": "PHONE_NUMBER",
        "Expected": "+94 77 123 4567",
        "Extracted": "+9477 123 4567",
        "Exact_Match": True
    }
])

display(final_validation)

final_validation.to_csv(
    "notebook06_entity_value_evaluation.csv",
    index=False
)

print("Entity value evaluation saved successfully.")
print(
    "Exact value matches:",
    final_validation["Exact_Match"].sum(),
    "/",
    len(final_validation)
)

In [ ]:
import os

filename = "notebook06_entity_value_evaluation.csv"

if os.path.exists(filename):
    print("✅ File exists!")
    print("File size:", os.path.getsize(filename), "bytes")
else:
    print("❌ File not found!")

In [ ]:
import pandas as pd

df = pd.read_csv("notebook06_entity_value_evaluation.csv")

display(df)

print("\nNumber of rows:", len(df))
print("Columns:", df.columns.tolist())

In [ ]:
from google.colab import files

files.download("notebook06_entity_value_evaluation.csv")

In [ ]:

test_text = """
Employee Name: Nimal Perera
Email: nimal.perera@example.com
Phone: +94 77 123 4567
NIC: 199512345678
"""

results = analyzer.analyze(
    text=test_text,
    language="en"
)

print("Detected Entities")
print("=" * 50)

for result in results:
    detected_value = test_text[result.start:result.end]

    print(f"Entity Type : {result.entity_type}")
    print(f"Value       : {detected_value}")
    print(f"Confidence  : {result.score:.2f}")
    print("-" * 50)


In [ ]:

from presidio_analyzer import Pattern, PatternRecognizer

nic_pattern = Pattern(
    name="Sri Lankan NIC",
    regex=r"\b(?:\d{9}[VvXx]|\d{12})\b",
    score=0.85
)

nic_recognizer = PatternRecognizer(
    supported_entity="NIC_SRI_LANKA",
    patterns=[nic_pattern],
    supported_language="en"
)

analyzer.registry.add_recognizer(nic_recognizer)

print("Sri Lankan NIC recognizer registered successfully!")


In [ ]:

import pandas as pd

entity_rows = []

for result in results:
    entity_rows.append({
        "Entity Type": result.entity_type,
        "Detected Value": test_text[result.start:result.end],
        "Start": result.start,
        "End": result.end,
        "Confidence": round(result.score, 3)
    })

entity_df = pd.DataFrame(entity_rows)

display(entity_df)

entity_df.to_csv(
    "notebook06_entity_detection_results.csv",
    index=False
)

print("Entity detection results saved.")


Entity Detection Evaluation

In [ ]:

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

# Ground-truth test cases
test_cases = [
    {
        "text": "Contact nimal.perera@example.com",
        "expected": {"EMAIL_ADDRESS"}
    },
    {
        "text": "Call +94 77 123 4567",
        "expected": {"PHONE_NUMBER"}
    },
    {
        "text": "NIC: 199512345678",
        "expected": {"NIC_SRI_LANKA"}
    },
    {
        "text": "No sensitive information here",
        "expected": set()
    }
]

evaluation_rows = []

for case in test_cases:
    detected = analyzer.analyze(
        text=case["text"],
        language="en"
    )

    predicted = {r.entity_type for r in detected}
    expected = case["expected"]

    evaluation_rows.append({
        "Text": case["text"],
        "Expected Entities": ", ".join(sorted(expected)) or "None",
        "Detected Entities": ", ".join(sorted(predicted)) or "None",
        "Exact Match": expected == predicted
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

# Entity-level evaluation across the test cases
all_labels = sorted(
    set().union(*(case["expected"] for case in test_cases),
                *(set(row["Detected Entities"].split(", "))
                  if row["Detected Entities"] != "None" else set()
                  for row in evaluation_rows))
)

y_true = []
y_pred = []

for case, row in zip(test_cases, evaluation_rows):
    expected = case["expected"]
    detected = (
        set(row["Detected Entities"].split(", "))
        if row["Detected Entities"] != "None" else set()
    )

    for label in all_labels:
        y_true.append(int(label in expected))
        y_pred.append(int(label in detected))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)

print("\nPreliminary Entity Detection Evaluation")
print("---------------------------------------")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"Exact matches: {evaluation_df['Exact Match'].sum()}/{len(evaluation_df)}")
print("\nNote: Results are based on only four manually defined test cases.")


In [ ]:

evaluation_df.to_csv(
    "notebook06_entity_evaluation.csv",
    index=False
)

print("Evaluation results saved successfully!")


False positives identify

In [ ]:

for _, row in evaluation_df.iterrows():
    expected = set(
        row["Expected Entities"].split(", ")
    ) if row["Expected Entities"] != "None" else set()

    detected = set(
        row["Detected Entities"].split(", ")
    ) if row["Detected Entities"] != "None" else set()

    print("\nText:", row["Text"])
    print("Expected:", expected or {"None"})
    print("Detected:", detected or {"None"})
    print("False positives:", detected - expected)
    print("Missed entities:", expected - detected)
    print("Exact match:", row["Exact Match"])


In [ ]:

import re

def detect_entities_cleaned(text):
    results = analyzer.analyze(
        text=text,
        language="en"
    )

    # Collect email spans to prevent URL overlap
    email_spans = [
        (r.start, r.end)
        for r in results
        if r.entity_type == "EMAIL_ADDRESS"
    ]

    cleaned = []

    for r in results:
        # Remove URL detections that overlap an email address
        if r.entity_type == "URL" and any(
            r.start < end and r.end > start
            for start, end in email_spans
        ):
            continue

        # Remove conflicting default US recognizers for Sri Lankan NIC
        if r.entity_type in {
            "US_BANK_NUMBER",
            "US_DRIVER_LICENSE",
            "PERSON"
        }:
            value = text[r.start:r.end]

            # Ignore these labels when they refer to a 12-digit NIC
            if re.fullmatch(r"\d{12}", value):
                continue

        cleaned.append(r)

    return cleaned


# Re-test the same examples
for case in test_cases:
    detected = detect_entities_cleaned(case["text"])
    predicted = {r.entity_type for r in detected}

    print("\nText:", case["text"])
    print("Expected:", case["expected"] or {"None"})
    print("Detected:", predicted or {"None"})
    print("False positives:", predicted - case["expected"])
    print("Missed entities:", case["expected"] - predicted)
    print("Exact match:", predicted == case["expected"])


In [ ]:

updated_rows = []

for case in test_cases:
    detected = detect_entities_cleaned(case["text"])
    predicted = {r.entity_type for r in detected}
    expected = case["expected"]

    updated_rows.append({
        "Text": case["text"],
        "Expected Entities": ", ".join(sorted(expected)) or "None",
        "Detected Entities": ", ".join(sorted(predicted)) or "None",
        "Exact Match": predicted == expected
    })

updated_df = pd.DataFrame(updated_rows)
display(updated_df)

all_labels = sorted(
    set().union(
        *(case["expected"] for case in test_cases),
        *(
            set(row["Detected Entities"].split(", "))
            if row["Detected Entities"] != "None" else set()
            for row in updated_rows
        )
    )
)

y_true, y_pred = [], []

for case, row in zip(test_cases, updated_rows):
    expected = case["expected"]
    predicted = (
        set(row["Detected Entities"].split(", "))
        if row["Detected Entities"] != "None" else set()
    )

    for label in all_labels:
        y_true.append(int(label in expected))
        y_pred.append(int(label in predicted))

p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)

print(f"Updated Precision: {p:.3f}")
print(f"Updated Recall:    {r:.3f}")
print(f"Updated F1-score:  {f1:.3f}")
print(f"Exact matches: {updated_df['Exact Match'].sum()}/{len(updated_df)}")

updated_df.to_csv("notebook06_entity_evaluation_improved.csv", index=False)


In [ ]:

# Save the improved evaluation results
updated_df.to_csv(
    "notebook06_entity_evaluation_improved.csv",
    index=False
)

# Save the evaluation metrics
metrics_df = pd.DataFrame([{
    "Precision": round(p, 3),
    "Recall": round(r, 3),
    "F1_Score": round(f1, 3),
    "Exact_Matches": f"{updated_df['Exact Match'].sum()}/{len(updated_df)}",
    "Test_Cases": len(updated_df)
}])

display(metrics_df)

metrics_df.to_csv(
    "notebook06_evaluation_metrics.csv",
    index=False
)

print("Evaluation results saved successfully!")


In [ ]:


import pandas as pd
from IPython.display import display

validation_df = pd.DataFrame([
    {
        "Entity": "PERSON",
        "Ground truth": "Nimal Perera",
        "OCR / extracted value": "Nimal Perera",
        "Interpretation": "Match"
    },
    {
        "Entity": "EMAIL_ADDRESS",
        "Ground truth": "nimal.perera@example.com",
        "OCR / extracted value": "nimalperera@example.com",
        "Interpretation": "Mismatch"
    },
    {
        "Entity": "PHONE_NUMBER",
        "Ground truth": "+94 77 123 4567",
        "OCR / extracted value": "+9477 123 4567",
        "Interpretation": "Digits match; formatting differs"
    }
])

display(validation_df)

exact_matches = sum(
    validation_df["Ground truth"] ==
    validation_df["OCR / extracted value"]
)

print(f"Exact-value matches: {exact_matches}/{len(validation_df)}")

validation_df.to_csv(
    "notebook06_entity_value_validation.csv",
    index=False
)


## Conclusion

The Entity Detection Engine was tested using six manually
labelled entity-type test cases. All six cases achieved an
exact match, with preliminary type-level precision, recall,
and F1-score of 1.0000.

*   List item
*   List item

OCR quality was evaluated using a synthetic test image.
The original OCR achieved a CER of 7.79% and a WER of 60.00%.
Image preprocessing reduced these values to 5.19% and 40.00%,
respectively.

Although person, email, and phone entities were detected
from the preprocessed OCR text, the extracted email value
contained an OCR error. Therefore, entity-type detection
success does not guarantee exact sensitive-value extraction.

These results are preliminary and are not representative
of overall framework performance. Further evaluation using
a larger, independently labelled dataset is required.